https://arxiv.org/pdf/2309.00325

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# 1. Data Preparation
# Load your HF and LF data
hf_data = np.load('hf_data.npy')  # Shape: (num_snapshots, spatial_dim)
lf_data = np.load('lf_data.npy')  # Shape: (num_snapshots, spatial_dim)

In [ ]:
version = 'vbnn1.11'
file_in=f'Ge77_rates_CNP_{version}.csv'
if not os.path.exists(f'out/{version}'):
   os.makedirs(f'out/{version}')
   

# Set parameter name/x_labels -> needs to be consistent with data input file
x_labels=['Radius[cm]','Thickness[cm]','NPanels', 'Theta[deg]', 'Length[cm]']
x_labels_out = ['Radius [cm]','Thickness [cm]','NPanels', 'Angle [deg]', 'Length [cm]']

y_err_label_cnp = 'Ge-77_CNP_err'
y_label_sim = 'rGe77[nuc/(kg*yr)]'

# Set parameter boundaries
xmin=[0,0,0,0,0]
xmax=[265,20,360,90,150]

# Set parameter boundaries for aquisition function
xlow=[90,2,4,0,1]
xhigh=[250,15,360,90,150]

# Assign costs
low_fidelity_cost = 1.
high_fidelity_cost = 2000.
n_fidelities = 2
# Set a fixed point in space for drawings
x_fixed = [160, 2, 40, 45, 20]
# number of sigma for error band drawing on prediction
factor=1.

# Get LF noise from file
#with open(f'in/{file_in}') as f:
#    first_line = f.readline()
#LF_noise=np.round(float(first_line.split(' +')[0].split('= ')[1]),3)
LF_noise = 0.028
# Get HF and LF data samples from file

data=pd.read_csv(f'in/{file_in}')
#data=data[[f'Mode', x_labels[0], x_labels[1], x_labels[2], x_labels[3], x_labels[4],y_label_cnp,y_err_label_cnp,y_label_sim]]


In [ ]:

x_train_l, x_train_h, y_train_l, y_train_h = ([],[],[],[])
row_h=data.index[data['Mode'] == 1]
row_l=data.index[data['Mode'] == 0]

hf_data_x = data.loc[data['Mode']==1.][x_labels].to_numpy()
hf_data_y = data.loc[data['Mode']==1.][y_label_sim].to_numpy()

lf_data_x = data.loc[data['Mode']==0.][x_labels].to_numpy()
lf_data_y = data.loc[data['Mode']==0.][ y_label_sim].to_numpy()

In [ ]:
# 2. Proper Orthogonal Decomposition (POD)
def compute_pod_basis(data, num_modes):
    # Subtract mean
    mean_data = np.mean(data, axis=0)
    data_centered = data - mean_data
    # Compute SVD
    U, S, Vt = np.linalg.svd(data_centered, full_matrices=False)
    # Select the first 'num_modes' modes
    basis = Vt[:num_modes].T
    return basis, mean_data

num_modes = 10  # Number of POD modes
pod_basis, mean_hf = compute_pod_basis(hf_data, num_modes)

In [ ]:
# 3. Projection onto Reduced Basis
def project_onto_basis(data, basis, mean_data):
    data_centered = data - mean_data
    reduced_states = np.dot(data_centered, basis)
    return reduced_states

hf_reduced_states = project_onto_basis(hf_data, pod_basis, mean_hf)
lf_reduced_states = project_onto_basis(lf_data, pod_basis, mean_hf)

In [ ]:
# 4. Multi-Fidelity LSTM Network
class MultiFidelityLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(MultiFidelityLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h_lstm, _ = self.lstm(x)
        out = self.fc(h_lstm[:, -1, :])
        return out

input_dim = lf_reduced_states.shape[1]
hidden_dim = 50
output_dim = hf_reduced_states.shape[1]
num_layers = 2

model = MultiFidelityLSTM(input_dim, hidden_dim, output_dim, num_layers)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Convert data to PyTorch tensors
lf_reduced_states_tensor = torch.tensor(lf_reduced_states, dtype=torch.float32)
hf_reduced_states_tensor = torch.tensor(hf_reduced_states, dtype=torch.float32)

# Training loop
num_epochs = 1000
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(lf_reduced_states_tensor.unsqueeze(1))
    loss = criterion(outputs, hf_reduced_states_tensor)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')



In [ ]:
# 5. Reconstruction
def reconstruct_from_basis(reduced_states, basis, mean_data):
    reconstructed_data = np.dot(reduced_states, basis.T) + mean_data
    return reconstructed_data

# Predict HF reduced states from new LF data
model.eval()
with torch.no_grad():
    new_lf_data = np.load('new_lf_data.npy')  # New LF data
    new_lf_reduced_states = project_onto_basis(new_lf_data, pod_basis, mean_hf)
    new_lf_reduced_states_tensor = torch.tensor(new_lf_reduced_states, dtype=torch.float32)
    predicted_hf_reduced_states = model(new_lf_reduced_states_tensor.unsqueeze(1)).numpy()

# Reconstruct HF data
predicted_hf_data = reconstruct_from_basis(predicted_hf_reduced_states, pod_basis, mean_hf)